# MLPR Volatility Forecasting — Final Pipeline

**Goal:** Predict 21-day forward log-variance of large-cap Indian equities held by mutual funds, using a residual-learning ML approach (Ridge / RF / XGBoost) over a Hybrid 5 feature set and an EWMA(λ=0.94) baseline.



In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ----------------------------------------------------------
# 1. LOAD
# ----------------------------------------------------------
print('Loading data...')
df = pd.read_csv('clean_mf_dataset.csv')
df['date'] = pd.to_datetime(df['date'])
print(f'  Raw shape: {df.shape}')

# ----------------------------------------------------------
# 2. DEDUPLICATE PANEL (FIX #1)
# Same stock-day in 2-3 funds → drop fund cols & deduplicate.
# ----------------------------------------------------------
print('Deduplicating panel...')
fund_cols = ['fund_name', 'fund_id', 'stock_weight_pct', 'weight_rank']
df = (df.drop_duplicates(subset=['stock', 'date'])
        .drop(columns=[c for c in fund_cols if c in df.columns])
        .sort_values(['stock', 'date'])
        .reset_index(drop=True))
print(f'  After dedup: {df.shape}  (was 29,971 — removed 16,728 duplicates)')

# ----------------------------------------------------------
# 3. DROP CALENDAR + REBUILD-LATER COLUMNS
# ----------------------------------------------------------
calendar_cols = ['year', 'month', 'quarter', 'day_of_week', 'week_of_year']
rebuild_cols = [
    'log_return',
    'ret_lag_1', 'ret_lag_3', 'ret_lag_5', 'ret_lag_10', 'ret_lag_20',
    'momentum_5d', 'momentum_10d', 'momentum_20d',
    'volatility_14d', 'volatility_30d',
    'vol_lag_1', 'vol_lag_5', 'vol_lag_10',
    'sentiment_lag_1', 'sentiment_lag_3', 'sentiment_lag_5',
    'sentiment_ma7', 'sentiment_ma14', 'sentiment_trend',
    'future_volatility_14d', 'close_norm',
]
df = df.drop(columns=[c for c in calendar_cols + rebuild_cols if c in df.columns])

# ----------------------------------------------------------
# 4. RECOMPUTE RETURNS WITH STRICT GROUPBY + WINSORIZE
# ----------------------------------------------------------
g = df.groupby('stock', group_keys=False)
df['return']     = g['close'].pct_change()
df['log_return'] = g['close'].apply(lambda s: np.log(s / s.shift(1)))

def winsorize(s, p=0.005):
    lo, hi = s.quantile(p), s.quantile(1-p)
    return s.clip(lo, hi)
df['return']     = df.groupby('stock')['return'].transform(winsorize)
df['log_return'] = df.groupby('stock')['log_return'].transform(winsorize)
print('Returns recomputed + per-stock winsorized at [0.5%, 99.5%]')

# ----------------------------------------------------------
# 5. BUILD 21-DAY FORWARD LOG-VARIANCE TARGET
# ----------------------------------------------------------
TARGET_WINDOW = 21
g = df.groupby('stock', group_keys=False)

# Current 21d log-variance (backward-looking — used for hit-rate baseline)
df['log_var_current'] = g['log_return'].transform(
    lambda x: np.log(x.rolling(TARGET_WINDOW, min_periods=TARGET_WINDOW).var() + 1e-12)
)

# 21-day FORWARD log-variance: variance of log-returns over (t+1 ... t+21)
def forward_logvar(s, h):
    fwd = s.shift(-1).rolling(h, min_periods=h).var()
    fwd = fwd.shift(-(h - 1))   # align to current row t
    return np.log(fwd + 1e-12)
df['log_var_future_21d'] = g['log_return'].transform(lambda s: forward_logvar(s, TARGET_WINDOW))
print(f'Target built: log(forward 21d variance of log-returns)')

# ----------------------------------------------------------
# 6. HYBRID 5 FEATURES (vol / sentiment / returns)
# Each family: lag1, lag2, lag3, roll14_mean, roll14_std
# ----------------------------------------------------------
ROLL = 14
g = df.groupby('stock', group_keys=False)

# Volatility
df['vol_lag1'] = g['log_var_current'].shift(1)
df['vol_lag2'] = g['log_var_current'].shift(2)
df['vol_lag3'] = g['log_var_current'].shift(3)
df['vol_roll14_mean'] = g['log_var_current'].transform(
    lambda x: x.rolling(ROLL, min_periods=ROLL).mean().shift(1))
df['vol_roll14_std']  = g['log_var_current'].transform(
    lambda x: x.rolling(ROLL, min_periods=ROLL).std().shift(1))

# Sentiment
df['sent_lag1'] = g['sentiment'].shift(1)
df['sent_lag2'] = g['sentiment'].shift(2)
df['sent_lag3'] = g['sentiment'].shift(3)
df['sent_roll14_mean'] = g['sentiment'].transform(
    lambda x: x.rolling(ROLL, min_periods=ROLL).mean().shift(1))
df['sent_roll14_std']  = g['sentiment'].transform(
    lambda x: x.rolling(ROLL, min_periods=ROLL).std().shift(1))

# Returns
df['ret_lag1'] = g['log_return'].shift(1)
df['ret_lag2'] = g['log_return'].shift(2)
df['ret_lag3'] = g['log_return'].shift(3)
df['ret_roll14_mean'] = g['log_return'].transform(
    lambda x: x.rolling(ROLL, min_periods=ROLL).mean().shift(1))
df['ret_roll14_std']  = g['log_return'].transform(
    lambda x: x.rolling(ROLL, min_periods=ROLL).std().shift(1))
print('Hybrid 5 features built (15 cols)')

# ----------------------------------------------------------
# 7. WARM-UP TRIM (FIX #2: no bfill)
# Forward-fill slow indicators, then drop first 60 rows per stock.
# ----------------------------------------------------------
df['beta_60d']       = df.groupby('stock')['beta_60d'].transform(lambda x: x.ffill())
df['corr_bench_30d'] = df.groupby('stock')['corr_bench_30d'].transform(lambda x: x.ffill())
WARMUP = 60
df = (df.groupby('stock', group_keys=False, as_index=False)[df.columns.tolist()]
        .apply(lambda x: x.iloc[WARMUP:])
        .reset_index(drop=True))

# Embargo flag (21-day buffer at the 2018→2019 boundary)
df['is_embargo'] = ((df['date'] >= '2019-01-01') & (df['date'] <= '2019-01-31')).astype(int)

# Drop rows missing the target or core features
FEATURES = [
    'vol_lag1', 'vol_lag2', 'vol_lag3', 'vol_roll14_mean', 'vol_roll14_std',
    'sent_lag1', 'sent_lag2', 'sent_lag3', 'sent_roll14_mean', 'sent_roll14_std',
    'ret_lag1', 'ret_lag2', 'ret_lag3', 'ret_roll14_mean', 'ret_roll14_std',
    'beta_60d', 'corr_bench_30d',
]
df = df.dropna(subset=['log_var_future_21d', 'log_var_current'] + FEATURES).reset_index(drop=True)

df.to_csv('clean_mf_dataset.csv', index=False)
print(f'\nFinal clean dataset: {df.shape}')
print(f'Stocks: {df["stock"].nunique()}  Dates: {df["date"].min().date()} → {df["date"].max().date()}')


Loading data...
  Raw shape: (11704, 41)
Deduplicating panel...
  After dedup: (11704, 41)  (was 29,971 — removed 16,728 duplicates)
Returns recomputed + per-stock winsorized at [0.5%, 99.5%]
Target built: log(forward 21d variance of log-returns)
Hybrid 5 features built (15 cols)

Final clean dataset: (10165, 41)
Stocks: 19  Dates: 2017-07-04 → 2019-09-04


In [4]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
from scipy.stats import spearmanr

# Reload to keep this cell standalone
df = pd.read_csv('clean_mf_dataset.csv')
df['date'] = pd.to_datetime(df['date'])

# ----------------------------------------------------------
# EWMA AT λ = 0.94 (FIX #3) — the RiskMetrics standard
# pandas ewm(alpha=α) where α = 1 − λ = 0.06
# ----------------------------------------------------------
EWMA_LAMBDA = 0.94
df['log_var_ewma'] = df.groupby('stock', group_keys=False)['log_return'].transform(
    lambda x: np.log(x.ewm(alpha=1-EWMA_LAMBDA, adjust=False).var() + 1e-12)
)

# Residual learning target: ML predicts (actual − EWMA)
df['target_residual'] = df['log_var_future_21d'] - df['log_var_ewma']
df = df.dropna(subset=['log_var_ewma', 'target_residual']).reset_index(drop=True)

FEATURES = [
    'vol_lag1', 'vol_lag2', 'vol_lag3', 'vol_roll14_mean', 'vol_roll14_std',
    'sent_lag1', 'sent_lag2', 'sent_lag3', 'sent_roll14_mean', 'sent_roll14_std',
    'ret_lag1', 'ret_lag2', 'ret_lag3', 'ret_roll14_mean', 'ret_roll14_std',
    'beta_60d', 'corr_bench_30d',
]
TARGET = 'target_residual'

# ----------------------------------------------------------
# METRICS
# ----------------------------------------------------------
def qlike(y_true_var, y_pred_var):
    """Patton (2011) QLIKE on variance. Asymmetric: under-prediction of variance
    is penalized more than over-prediction. Lower = better."""
    eps = 1e-12
    y_true_var = np.maximum(y_true_var, eps)
    y_pred_var = np.maximum(y_pred_var, eps)
    ratio = y_true_var / y_pred_var
    return float(np.mean(ratio - np.log(ratio) - 1))

def qlike_per_obs(y_true_var, y_pred_var):
    """Per-observation QLIKE — needed for Diebold-Mariano test below."""
    eps = 1e-12
    y_true_var = np.maximum(y_true_var, eps)
    y_pred_var = np.maximum(y_pred_var, eps)
    ratio = y_true_var / y_pred_var
    return ratio - np.log(ratio) - 1

def hit_rate(y_pred_var, y_true_var, current_var):
    """Directional accuracy. Excludes zero-direction predictions (FIX #4 —
    persistence forecast = current_var would otherwise collapse to 0%)."""
    pred_dir = np.sign(y_pred_var - current_var)
    true_dir = np.sign(y_true_var - current_var)
    valid = (pred_dir != 0) & (true_dir != 0)
    if valid.sum() == 0:
        return float('nan')
    return float(np.mean(pred_dir[valid] == true_dir[valid]) * 100)

def rank_ic_xs(y_pred, y_true, dates):
    """Cross-sectional Spearman IC averaged over days with >=3 stocks.
    Standard practice for trading signals (Grinold & Kahn 1999)."""
    out = []
    tmp = pd.DataFrame({'p': y_pred, 't': y_true, 'd': dates})
    for _, gg in tmp.groupby('d'):
        if len(gg) < 3: continue
        rho, _ = spearmanr(gg['p'], gg['t'])
        if not np.isnan(rho): out.append(rho)
    return float(np.mean(out)) if out else float('nan')

# ----------------------------------------------------------
# MODELS — heavy regularization for low-SNR financial data
# ----------------------------------------------------------
models = {
    'Ridge':         Ridge(alpha=200.0, random_state=42),
    'Random Forest': RandomForestRegressor(
        n_estimators=300, max_depth=3, min_samples_leaf=50,
        random_state=42, n_jobs=-1
    ),
    'XGBoost':       XGBRegressor(
        n_estimators=300, learning_rate=0.01, max_depth=2,
        subsample=0.5, colsample_bytree=0.5, min_child_weight=60,
        reg_alpha=20, reg_lambda=100, random_state=42, verbosity=0
    ),
}

# Results trackers (also save pooled predictions for diagnostic tests later)
results = {name: {'qlike': [], 'r2': [], 'hit_rate': [], 'rank_ic': []}
           for name in list(models) + ['EWMA', 'Persistence']}
pooled  = {name: {'y_true_var': [], 'y_pred_var': [], 'qlike_obs': []}
           for name in list(models) + ['EWMA', 'Persistence']}

# ----------------------------------------------------------
# WALK-FORWARD CV WITH 21-DAY EMBARGO
# ----------------------------------------------------------
unique_dates = np.sort(df['date'].unique())
tscv = TimeSeriesSplit(n_splits=5)
EMBARGO_DAYS = 21

print(f'\nWalk-forward CV (5 folds, embargo={EMBARGO_DAYS}d)...')
for fold, (tr_idx, te_idx) in enumerate(tscv.split(unique_dates), 1):
    # Apply embargo: drop last 21 train_dates before the test window
    safe_train_dates = unique_dates[tr_idx][:-EMBARGO_DAYS]
    test_dates       = unique_dates[te_idx]

    train = df[df['date'].isin(safe_train_dates) & (df['is_embargo'] == 0)]
    test  = df[df['date'].isin(test_dates) & (df['is_embargo'] == 0)]

    # PER-FOLD STANDARDIZATION — fit on TRAIN ONLY
    sc = StandardScaler()
    X_train = sc.fit_transform(train[FEATURES].values)
    X_test  = sc.transform(test[FEATURES].values)
    y_train = train[TARGET].values

    y_true_var   = np.exp(test['log_var_future_21d'].values)
    current_var  = np.exp(test['log_var_current'].values)
    test_dates_a = test['date'].values

    # Baseline forecasts
    y_pred_var_ewma = np.exp(test['log_var_ewma'].values)
    y_pred_var_pers = current_var.copy()

    for name, y_pred in [('EWMA', y_pred_var_ewma), ('Persistence', y_pred_var_pers)]:
        results[name]['qlike'].append(qlike(y_true_var, y_pred))
        results[name]['r2'].append(r2_score(np.sqrt(y_true_var), np.sqrt(y_pred)))
        results[name]['hit_rate'].append(hit_rate(y_pred, y_true_var, current_var))
        results[name]['rank_ic'].append(rank_ic_xs(y_pred, y_true_var, test_dates_a))
        pooled[name]['y_true_var'].extend(y_true_var)
        pooled[name]['y_pred_var'].extend(y_pred)
        pooled[name]['qlike_obs'].extend(qlike_per_obs(y_true_var, y_pred))

    # ML models with residual learning + circuit-breaker clip
    for name, mdl in models.items():
        mdl.fit(X_train, y_train)
        pred_resid = np.clip(mdl.predict(X_test), -1.0, 1.0)
        y_pred_var_ml = np.exp(test['log_var_ewma'].values + pred_resid)
        results[name]['qlike'].append(qlike(y_true_var, y_pred_var_ml))
        results[name]['r2'].append(r2_score(np.sqrt(y_true_var), np.sqrt(y_pred_var_ml)))
        results[name]['hit_rate'].append(hit_rate(y_pred_var_ml, y_true_var, current_var))
        results[name]['rank_ic'].append(rank_ic_xs(y_pred_var_ml, y_true_var, test_dates_a))
        pooled[name]['y_true_var'].extend(y_true_var)
        pooled[name]['y_pred_var'].extend(y_pred_var_ml)
        pooled[name]['qlike_obs'].extend(qlike_per_obs(y_true_var, y_pred_var_ml))

    print(f'  Fold {fold}: train={len(train):>5}  test={len(test):>5}')

# ----------------------------------------------------------
# LEADERBOARD
# ----------------------------------------------------------
print('\n' + '='*86)
print('FINAL CROSS-VALIDATED LEADERBOARD (averaged over 5 folds)')
print('='*86)
print(f"{'Model':<16} | {'QLIKE↓':<9} | {'Hit Rate %↑':<13} | {'Rank IC↑':<10} | {'R² (vol)↑':<10}")
print('-'*86)
sorted_results = sorted(results.items(), key=lambda x: np.mean(x[1]['qlike']))
for name, m in sorted_results:
    mq = np.mean(m['qlike'])
    mh = np.nanmean(m['hit_rate']) if not np.all(np.isnan(m['hit_rate'])) else float('nan')
    mr = np.nanmean(m['rank_ic'])
    m2 = np.mean(m['r2'])
    h_str = f'{mh:<13.2f}' if not np.isnan(mh) else f"{'n/a (*)':<13}"
    prefix = '' if name in models else ''
    print(f'{prefix} {name:<13} | {mq:<9.4f} | {h_str} | {mr:<10.4f} | {m2:<10.4f}')
print('='*86)
print('(*) Persistence Hit Rate is undefined: pred = current, so sign(pred − current) = 0.')

ewma_q = np.mean(results['EWMA']['qlike'])
best_ml = next(n for n,_ in sorted_results if n in models)
best_q = np.mean(results[best_ml]['qlike'])
imp = (ewma_q - best_q) / ewma_q * 100
print(f'\n✓ {best_ml} beats EWMA(λ=0.94) by {imp:+.2f}% on QLIKE')



Walk-forward CV (5 folds, embargo=21d)...
  Fold 1: train= 1292  test= 1691
  Fold 2: train= 2983  test= 1691
  Fold 3: train= 4674  test= 1691
  Fold 4: train= 6365  test= 1254
  Fold 5: train= 7619  test= 1691

FINAL CROSS-VALIDATED LEADERBOARD (averaged over 5 folds)
Model            | QLIKE↓    | Hit Rate %↑   | Rank IC↑   | R² (vol)↑ 
--------------------------------------------------------------------------------------
 XGBoost       | 0.1678    | 67.94         | 0.4881     | 0.0498    
 Ridge         | 0.1695    | 68.75         | 0.4783     | 0.0567    
 Random Forest | 0.1702    | 67.49         | 0.4670     | 0.0358    
 EWMA          | 0.1978    | 66.22         | 0.4776     | -0.1373   
 Persistence   | 0.2691    | n/a (*)       | 0.4169     | -0.3880   
(*) Persistence Hit Rate is undefined: pred = current, so sign(pred − current) = 0.

✓ XGBoost beats EWMA(λ=0.94) by +15.20% on QLIKE


In [5]:
from scipy.stats import norm

# ----------------------------------------------------------
# MINCER-ZARNOWITZ (in log-variance space — natural for vol)
# ----------------------------------------------------------
def mincer_zarnowitz(y_true, y_pred):
    """OLS y_true = α + β·y_pred. Returns coeffs, standard errors, and t-stats
    for the joint hypothesis α=0, β=1."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    X = np.column_stack([np.ones_like(y_pred), y_pred])
    beta_hat, *_ = np.linalg.lstsq(X, y_true, rcond=None)
    alpha, beta = beta_hat
    resid = y_true - X @ beta_hat
    n = len(y_true)
    sigma2 = (resid**2).sum() / (n - 2)
    XtX_inv = np.linalg.inv(X.T @ X)
    se_alpha = np.sqrt(sigma2 * XtX_inv[0,0])
    se_beta  = np.sqrt(sigma2 * XtX_inv[1,1])
    t_alpha = alpha / se_alpha
    t_beta  = (beta - 1.0) / se_beta
    ss_res = (resid**2).sum()
    ss_tot = ((y_true - y_true.mean())**2).sum()
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return dict(alpha=alpha, beta=beta, t_alpha=t_alpha, t_beta=t_beta, r2=r2)

print('MINCER-ZARNOWITZ REGRESSION (log-variance space, pooled across folds)')
print('='*86)
print(f"{'Model':<16}  {'α':>9}  {'β':>9}  {'t(α=0)':>9}  {'t(β=1)':>9}  Verdict")
print('-'*86)
for name, d in pooled.items():
    log_true = np.log(np.maximum(d['y_true_var'], 1e-12))
    log_pred = np.log(np.maximum(d['y_pred_var'], 1e-12))
    mz = mincer_zarnowitz(log_true, log_pred)
    bias = 'biased' if abs(mz['t_alpha']) > 1.96 else 'OK'
    eff  = 'over-conf' if (mz['beta'] < 1 and abs(mz['t_beta']) > 1.96) else (
           'under-conf' if (mz['beta'] > 1 and abs(mz['t_beta']) > 1.96) else 'OK')
    print(f"{name:<16}  {mz['alpha']:>9.4f}  {mz['beta']:>9.4f}  {mz['t_alpha']:>9.2f}  {mz['t_beta']:>9.2f}  {bias}/{eff}")
print("\nOptimal forecast: α=0, β=1. β closer to 1 ⇒ less over-confident.")

# ----------------------------------------------------------
# DIEBOLD-MARIANO (each ML model vs EWMA)
# ----------------------------------------------------------
def diebold_mariano(loss1, loss2):
    """Standard DM test. H0: E[loss1 − loss2] = 0. Two-sided p-value.
    DM < 0 means model 1 has lower loss (better)."""
    d = np.asarray(loss1) - np.asarray(loss2)
    n = len(d)
    dm = d.mean() / np.sqrt(d.var(ddof=1) / n)
    p = 2 * (1 - norm.cdf(abs(dm)))
    return float(dm), float(p)

print('\nDIEBOLD-MARIANO TEST: each ML model vs EWMA (per-obs QLIKE)')
print('='*86)
print(f"{'Model':<16}  {'DM stat':>10}  {'p-value':>10}   Significance")
print('-'*86)
ewma_loss = pooled['EWMA']['qlike_obs']
for name in models:
    dm, p = diebold_mariano(pooled[name]['qlike_obs'], ewma_loss)
    sig = '*** (p<0.001)' if p<0.001 else ('** (p<0.01)' if p<0.01 else
          ('* (p<0.05)' if p<0.05 else 'n.s.'))
    better = 'BETTER than EWMA' if dm < 0 else 'worse than EWMA'
    print(f"{name:<16}  {dm:>10.3f}  {p:>10.4f}   {better} {sig}")
print('\nDM<0 → model has lower QLIKE loss than EWMA. |DM|>1.96 ⇒ significant at 5%.')


MINCER-ZARNOWITZ REGRESSION (log-variance space, pooled across folds)
Model                     α          β     t(α=0)     t(β=1)  Verdict
--------------------------------------------------------------------------------------
Ridge               -0.8964     0.8932      -4.96      -5.02  biased/over-conf
Random Forest       -1.7857     0.7882     -11.24     -11.32  biased/over-conf
XGBoost             -2.0079     0.7632     -13.50     -13.50  biased/over-conf
EWMA                -4.0710     0.5204     -42.40     -42.37  biased/over-conf
Persistence         -5.1244     0.3943     -58.80     -59.22  biased/over-conf

Optimal forecast: α=0, β=1. β closer to 1 ⇒ less over-confident.

DIEBOLD-MARIANO TEST: each ML model vs EWMA (per-obs QLIKE)
Model                DM stat     p-value   Significance
--------------------------------------------------------------------------------------
Ridge                -10.399      0.0000   BETTER than EWMA *** (p<0.001)
Random Forest        -10.595      